# Week 3 — Model Training & Selection

Predict daily demand using ML models.

In [1]:
import os
os.chdir(r'C:\Users\Chenna Keshava\Restaurant-demand-forecasting-and-inventory-optimization-Ai\notebooks')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [3]:
X_train = pd.read_csv('X_train.csv', index_col=0, parse_dates=True)
X_test  = pd.read_csv('X_test.csv',  index_col=0, parse_dates=True)

y_train = pd.read_csv('y_train.csv', index_col=0, parse_dates=True).squeeze()
y_test  = pd.read_csv('y_test.csv',  index_col=0, parse_dates=True).squeeze()

X_train = X_train.select_dtypes(include=[np.number])
X_test  = X_test.select_dtypes(include=[np.number])

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

Train Shape: (888, 15)
Test Shape : (222, 15)


## Evaluation Function

In [4]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f"{name}")
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print("-"*30)

    return {"model": name, "MAE": mae, "RMSE": rmse}

results = []

## Linear Regression (Baseline)

In [5]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

results.append(evaluate("Linear Regression", y_test, y_pred_lr))

Linear Regression
MAE  : 438.92
RMSE : 716.25
------------------------------


## Random Forest

In [6]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

results.append(evaluate("Random Forest", y_test, y_pred_rf))

Random Forest
MAE  : 388.57
RMSE : 588.65
------------------------------


## XGBoost

In [7]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

results.append(evaluate("XGBoost", y_test, y_pred_xgb))

XGBoost
MAE  : 378.28
RMSE : 611.25
------------------------------


## Model Comparison

In [8]:
results_df = pd.DataFrame(results).sort_values(by="MAE")
results_df

,model,MAE,RMSE
2,XGBoost,378.277632,611.253144
1,Random Forest,388.572894,588.648100
0,Linear Regression,438.923574,716.252268


## TimeSeriesSplit

In [9]:
tscv = TimeSeriesSplit(n_splits=5)

## Hyperparameter Tuning

In [10]:
best_mae = float('inf')
best_params = {}

for max_depth in [3, 5, 7]:
    for lr_val in [0.01, 0.05, 0.1]:
        for n_est in [300, 500]:

            fold_maes = []

            for train_idx, val_idx in tscv.split(X_train):
                X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
                y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

                model = XGBRegressor(
                    max_depth=max_depth,
                    learning_rate=lr_val,
                    n_estimators=n_est,
                    random_state=42,
                    verbosity=0
                )

                model.fit(X_tr, y_tr)

                preds = model.predict(X_val)
                fold_maes.append(mean_absolute_error(y_val, preds))

            avg_mae = np.mean(fold_maes)

            if avg_mae < best_mae:
                best_mae = avg_mae
                best_params = {
                    "max_depth": max_depth,
                    "learning_rate": lr_val,
                    "n_estimators": n_est
                }

print("Best MAE:", best_mae)
print("Best Params:", best_params)

Best MAE: 482.94688027871615
Best Params: {'max_depth': 3, 'learning_rate': 0.01, 'n_estimators': 500}


## Final Model

In [11]:
xgb_best = XGBRegressor(**best_params, random_state=42, verbosity=0)
xgb_best.fit(X_train, y_train)

y_pred_best = xgb_best.predict(X_test)

results.append(evaluate("XGBoost Tuned", y_test, y_pred_best))

XGBoost Tuned
MAE  : 424.43
RMSE : 641.21
------------------------------


## Save Model

In [12]:
import os

os.makedirs('models', exist_ok=True)

In [14]:
import joblib

joblib.dump(xgb_best, 'models/best_model_xgb.pkl')

print("✅ Model saved ")

✅ Model saved 
